In [ ]:
import json,subprocess,sys,time,hashlib,zipfile,os,urllib.request,collections
from pathlib import Path
ROOT=Path('/kaggle/working')
BASE='https://github.com/TryDotAtwo/MultiGPUBFS/releases/download/native-pip-20261015-v16/'
receipt=json.loads(urllib.request.urlopen(BASE+'manifest.json',timeout=120).read())
assert receipt['sha256']=='65941655242e7a48fa3f27989992fe5f8d5d898c716d481ce0c92cdd9d4a5a6b'
wheel=Path('/tmp')/receipt['wheel'];urllib.request.urlretrieve(BASE+receipt['wheel'],wheel)
assert hashlib.sha256(wheel.read_bytes()).hexdigest()==receipt['sha256']
subprocess.run([sys.executable,'-m','pip','install','--no-deps','--force-reinstall',str(wheel)],check=True)
subprocess.run([sys.executable,'-m','pip','install','cayleypy==0.2.0'],check=True)

# Validate the ordinary resolver and normal user-facing API before deep contrasts.
import math,importlib.metadata,re
import multigpubfs
from cayleypy import CayleyGraph,PermutationGroups
from multigpubfs.native_distribution import native_runtime
from multigpubfs.generic_session import persistent_graph_workers
native,environment=native_runtime();native_root=Path(native).parents[1]
native_manifest=json.loads((native_root/'manifest.json').read_text())
assert native_manifest['production_build']=='shared-sm120'
assert native_manifest['cuda_runtime_linkage']=='Shared' and 120 in native_manifest['architectures']
assert 'diagnostic_variants' not in environment['LD_LIBRARY_PATH']
assert 'libcudart.so.12' in subprocess.check_output(['readelf','-d',str(native_root/'lib/libmgbfs_cuda.so')],text=True)
from multigpubfs.runtime_variants import inventory as build_inventory
builds=build_inventory();production=next(x for x in builds['variants'] if x['id']=='shared-sm120')
modules=collections.defaultdict(set)
for match in re.finditer(r'(\S+)\.sm_(\d+)\.cubin',production['build_receipt']['sass_inventory']):modules[match[1]].add(int(match[2]))
assert len(modules)==24 and all({75,86,120}<=values for values in modules.values())
api=[]
for devices in ((0,),(0,1)):
 multigpubfs.enable_native(multigpubfs.NativeOptions(devices=devices,max_seconds=120,output_root=ROOT/'api-results'))
 with persistent_graph_workers():
  for n in ((3,5,7,11,12) if len(devices)==2 else (3,5,7)):
   began=time.monotonic();graph=CayleyGraph(PermutationGroups.coxeter(n),device='cpu');result=graph.bfs()
   assert result.backend=='native' and result.bfs_completed
   assert result.num_vertices()==math.factorial(n) and result.diameter()==n*(n-1)//2
   expected=[1]
   for width in range(1,n+1):
    values=[0]*(len(expected)+width-1)
    for i,value in enumerate(expected):
     for shift in range(width):values[i+shift]+=value
    expected=values
   assert result.layer_sizes==expected
   api.append({'n':n,'devices':list(devices),'vertices':result.num_vertices(),'diameter':result.diameter(),'wall_seconds':time.monotonic()-began,'native_report_identity':{k:result.report[k] for k in ('status','world_size','devices','backend','n_ranks','device_count') if k in result.report}})
   print('USER_API_PASS',json.dumps(api[-1]),flush=True)
multigpubfs.disable_native()
(ROOT/'api-acceptance.json').write_text(json.dumps(api,indent=2))

started=time.monotonic()
q=subprocess.run([sys.executable,'-m','multigpubfs.diagnostics','--deep','--total-timeout','1800','--output',str(ROOT/'deep')],timeout=1960)
report=json.loads((ROOT/'deep/report.json').read_text());checks=json.loads((ROOT/'deep/manifest.json').read_text())
for entry in checks['files']:
 file=ROOT/'deep'/entry['path'];assert file.stat().st_size==entry['size'];assert hashlib.sha256(file.read_bytes()).hexdigest()==entry['sha256']
with zipfile.ZipFile(ROOT/'deep/diagnostics.zip') as z:
 assert 'deep-summary.txt' in z.namelist() and 'deep-analysis.json' in z.namelist()
 assert not any(Path(n).name=='states.json' or Path(n).name=='case-input.json' or Path(n).name.startswith('job-') for n in z.namelist())
summary={'schema':1,'status':'PASS' if report['status']=='PASS' and q.returncode==0 else 'FAIL','wheel':receipt['wheel'],'wheel_sha256':receipt['sha256'],'package':report['package'],'hardware':'2xT4','wall_seconds':time.monotonic()-started,'counts':dict(collections.Counter(s['status'] for s in report['steps'])),'runtime_build_matrix':report['deep_analysis']['runtime_build_matrix'],'scope':'Installed shared-SM120 production library executed on its native SM75 images on two T4 GPUs; no SM120 hardware or original-server acceptance claim','steps':[{k:s[k] for k in ('name','status','seconds','reason','error','last_native_phase') if k in s} for s in report['steps']]}
(ROOT/'sm120-production-acceptance.json').write_text(json.dumps(summary,indent=2))
assert report['package']=='0.2.0.dev16'
assert report['devices']==[0,1]
assert report['production_bfs_acceptance']['status']=='PASS'
assert report['production_bfs_acceptance']['build']=='shared-sm120'
summary.update(user_api=api,production_bfs_acceptance=report['production_bfs_acceptance'],production_library_sha256=native_manifest['sha256']['lib/libmgbfs_cuda.so'],production_architectures=native_manifest['architectures'],compiled_modules={k:sorted(v) for k,v in modules.items()})
matrix=report['deep_analysis']['runtime_build_matrix'];assert not matrix['root_cause_confirmed']
assert len(matrix['comparisons'])==6
assert all(c['outcome']=='ALL_PASS' for c in matrix['comparisons']),matrix
steps=[s for s in report['steps'] if s.get('diagnostic_case',{}).get('variant')]
assert len(steps)==24 and all(s['status']=='PASS' for s in steps)
for step in steps:
 assert step['partial_evidence']['variant.json']['id']==step['diagnostic_case']['variant']
 trace=(ROOT/'deep'/step['name']/'native-startup.log').read_text()
 for marker in ('hardware-cudaSetDevice-enter','hardware-cudaSetDevice-return','hardware-cudaGetDeviceProperties-enter','hardware-cudaGetDeviceProperties-return'):assert marker in trace,(step['name'],marker)
 if step['diagnostic_case'].get('stage')=='persistent':
  snapshot=step['partial_evidence']['process-snapshot.json'];library=step['partial_evidence']['variant.json']['resolved_libraries']['libmgbfs_cuda.so']
  workers=[p for p in snapshot['processes'] if Path(p['executable']).name=='mgbfs']
  assert workers and any(library in '\n'.join(p['loaded_libraries']) for p in workers),(step['name'],library)
for step in report['steps']:
 if step.get('pid'):
  f=Path('/proc')/str(step['pid'])/'stat'
  assert not f.exists() or f.read_text().rsplit(')',1)[1].split()[0]=='Z'
assert all(x['complete'] for x in report['diagnostic_cleanup'])
assert summary['status']=='PASS',[(s['name'],s.get('error')) for s in report['steps'] if s['status'] in ('FAIL','TIMEOUT')]
# Exercise the alternate library on real multi-GPU coverage paths too.
from multigpubfs.diagnostics import Runner
from multigpubfs.deep_diagnostics import run_case
coverage=Runner(ROOT/'coverage-delta',60);coverage.deep_capture=True;limit=time.monotonic()+240
for label,changes in [('multi-generic',{}),('matrix',{'fixture':'matrix'}),('multi-shard-ab-hash',{'fixture':'lrx','backend':'shard_ab_hash'}),('multi-shard-ab-sort',{'fixture':'lrx','backend':'shard_ab_sort_merge'})]:
 config={'kind':'bfs','fixture':'permutation','devices':[0,1],'coverage_build':True};config.update(changes)
 run_case(coverage,{'name':'coverage-'+label,'config':config,'seconds':45},limit)
coverage_result=coverage.finish();assert coverage_result['status']=='PASS'
summary['diagnostic_coverage_build']=report['diagnostic_coverage_build']
assert report['diagnostic_coverage_build']['id']=='shared-sm120'
summary['production_multi_gpu_validation']={'status':coverage_result['status'],'build':'shared-sm120','steps':[{k:s[k] for k in ('name','status','seconds') if k in s} for s in coverage_result['steps']],'scope':'Actual multi-GPU generic/matrix/specialized oracle probes on the normal production library'}
subprocess.run([sys.executable,'-m','multigpubfs.diagnostics','--output',str(ROOT/'normal')],check=True,timeout=180)
budget=subprocess.run([sys.executable,'-m','multigpubfs.diagnostics','--deep','--total-timeout','1','--output',str(ROOT/'budget')],timeout=30)
assert budget.returncode==1 and json.loads((ROOT/'budget/report.json').read_text())['status']=='INCOMPLETE'
assert (ROOT/'budget/diagnostics.zip').is_file()
summary.update(ordinary_doctor_status=json.loads((ROOT/'normal/report.json').read_text())['status'],budget_status='INCOMPLETE',archive_checksums_verified=True,owned_process_groups_reaped=True)
(ROOT/'sm120-production-acceptance.json').write_text(json.dumps(summary,indent=2))
print('SHARED_SM120_PRODUCTION_ACCEPTANCE_PASS',json.dumps(summary),flush=True)
